# muon_db Visualization Sandbox

This notebook visualizes the curated lakehouse tables. It focuses on the data-engineering outputs, not model training.

Recommended tables for plotting:

- `silver_muon`: cleaned muon kinematics and quality fields.
- `jet`: curated jet kinematics and b-tag score.
- `met`: event-level missing transverse energy.
- `event_summary`: event-level counts and global features.
- `dimuon`: opposite-sign dimuon mass and angular separation.

In [ ]:
from pathlib import Path
import sys

WORKSPACE = Path.cwd()
if WORKSPACE.name == "notebooks":
    WORKSPACE = WORKSPACE.parent
sys.path.insert(0, str(WORKSPACE))

import matplotlib.pyplot as plt
import mplhep as hep
import numpy as np

from src.access.muon_db import connect_with_tables

plt.style.use(hep.style.CMS)
connection, tables = connect_with_tables(WORKSPACE / "data" / "muon_db")

## Utility Functions

In [ ]:
def values(query: str, column: str = "value"):
    array = connection.execute(query).fetchnumpy()[column]
    array = np.asarray(array, dtype=float)
    return array[np.isfinite(array)]


def hist(values_array, bins, value_range, xlabel, title, ylabel="Count"):
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.hist(values_array, bins=bins, range=value_range, histtype="stepfilled", alpha=0.75)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(alpha=0.2)
    fig.tight_layout()
    return fig, ax

## Cleaned Muon pT

This uses `silver_muon`, so it reflects the applied physics object cleaning: tight ID, isolation below 0.15, and pT above 20 GeV.

In [ ]:
muon_pt = values("SELECT pt AS value FROM silver_muon WHERE pt IS NOT NULL")
hist(muon_pt, bins=80, value_range=(0, 200), xlabel="Muon pT [GeV]", title="muon_db: cleaned muon pT")

## Opposite-Sign Dimuon Mass

This uses the gold `dimuon` table. Each row is an opposite-sign cleaned muon pair with an invariant mass calculation.

In [ ]:
dimuon_mass = values("SELECT invariant_mass AS value FROM dimuon WHERE invariant_mass IS NOT NULL")
hist(dimuon_mass, bins=90, value_range=(0, 180), xlabel="m(mu, mu) [GeV]", title="muon_db: opposite-sign dimuon mass", ylabel="Pair count")

## MET Distribution

This uses the gold `met` table, one row per cleaned event.

In [ ]:
met_pt = values("SELECT met_pt AS value FROM met WHERE met_pt IS NOT NULL")
hist(met_pt, bins=80, value_range=(0, 250), xlabel="MET pT [GeV]", title="muon_db: MET", ylabel="Event count")

## Event-Level HT And ST

`HT` is the scalar sum of jet pT. `ST` is `HT + cleaned muon pT sum + MET`.

In [ ]:
event_features = connection.execute("SELECT HT, ST FROM event_summary").fetchdf()

fig, ax = plt.subplots(figsize=(8, 6))
ax.hist(event_features["HT"], bins=70, range=(0, 800), histtype="step", linewidth=2, label="HT")
ax.hist(event_features["ST"], bins=70, range=(0, 1000), histtype="step", linewidth=2, label="ST")
ax.set_xlabel("Energy scale [GeV]")
ax.set_ylabel("Event count")
ax.set_title("muon_db: event-level HT and ST")
ax.grid(alpha=0.2)
ax.legend()
fig.tight_layout()
fig

## Jet Multiplicity Vs MET

This plot uses `event_summary` to compare global event quantities after event-level cleaning.

In [ ]:
summary = connection.execute("SELECT n_jets, MET_pt, ST FROM event_summary WHERE MET_pt IS NOT NULL").fetchdf()

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(summary["n_jets"], summary["MET_pt"], c=summary["ST"], s=12, alpha=0.55)
ax.set_xlabel("Jet multiplicity")
ax.set_ylabel("MET pT [GeV]")
ax.set_title("muon_db: jet multiplicity vs MET")
ax.grid(alpha=0.2)
fig.colorbar(scatter, ax=ax, label="ST [GeV]")
fig.tight_layout()
fig

## Save A Figure

Use this pattern to save any plot from the notebook to `outputs/`.

In [ ]:
output = WORKSPACE / "outputs" / "notebook_dimuon_mass.png"
fig, ax = hist(dimuon_mass, bins=90, value_range=(0, 180), xlabel="m(mu, mu) [GeV]", title="muon_db: opposite-sign dimuon mass", ylabel="Pair count")
fig.savefig(output, dpi=160)
output